# HydraY NNUE — shakedown training su Colab (T4)

Runtime → Cambia tipo di runtime → **GPU (T4)**.

Prerequisito: carica su Google Drive il file dati interleaved
(`hydray_16M_interleaved.bin`, o la versione `.zst`) generato dal datagen.
Poi esegui le celle in ordine. Tempo previsto shakedown (10 superbatch): ~30-60 min.

In [ ]:
!nvidia-smi
!nvcc --version || ls /usr/local/cuda/bin

In [ ]:
# Rust toolchain
!curl --proto '=https' --tlsv1.2 -sSf https://sh.rustup.rs | sh -s -- -y --profile minimal
!~/.cargo/bin/cargo --version

In [ ]:
# Trainer HydraY (solo la cartella nnue/trainer serve)
!git clone --depth 1 --branch Claudio1.3.0 https://github.com/ThomasGhione/chess_engine
%cd chess_engine/nnue/trainer

In [ ]:
# Dati da Google Drive (adatta il path al tuo Drive)
from google.colab import drive
drive.mount('/content/drive')
import os, glob
src = glob.glob('/content/drive/MyDrive/hydray_16M_interleaved.bin*')[0]
print('trovato:', src)
if src.endswith('.zst'):
    !apt-get -qq install -y zstd
    !zstd -d -f "{src}" -o /content/data.bin
else:
    !cp "{src}" /content/data.bin
print(os.path.getsize('/content/data.bin') / 32, 'posizioni')

In [ ]:
# Training shakedown: 10 superbatch (~1B sample). Loss deve SCENDERE in modo netto.
# In coda stampa i sanity eval (startpos ~ +20..50 cp, coppia mirror identica, ±donna enorme).
!PATH=$HOME/.cargo/bin:$PATH CUDA_PATH=/usr/local/cuda cargo run -r --bin trainer --features cuda -- /content/data.bin 10 hydray-v1-shakedown

In [ ]:
# Verifica standalone del file quantizzato (stesso lettore che fara' da spec al loader C++)
!ls checkpoints/
!PATH=$HOME/.cargo/bin:$PATH cargo run -r --bin sanity -- checkpoints/hydray-v1-shakedown-10/quantised.bin

In [ ]:
# Scarica il checkpoint (quantised.bin ~400 KB)
from google.colab import files
files.download('checkpoints/hydray-v1-shakedown-10/quantised.bin')